In [18]:
import pandas as pd
import re
import zstandard as zstd
import io
import orjson
from collections import defaultdict
import json
from tqdm import tqdm
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
import matplotlib.pyplot as plt
import csv
import polars as pl
import os

In [15]:
top500 = pd.read_csv('Top500.csv')
top500.head()

,Rank,Title,Description,Users,Title_URL
0,1,r/funny,Reddit's largest humor depository,67M members,https://www.reddit.com/r/funny/
1,2,r/AskReddit,r/AskReddit is the place to ask and answer tho...,58M\n members,https://www.reddit.com/r/AskReddit/
2,3,r/worldnews,"A place for major news from around the world, ...",47M\n members,https://www.reddit.com/r/worldnews/
3,4,r/gaming,The Number One Gaming forum on the Internet.,47M\n members,https://www.reddit.com/r/gaming/
4,5,r/todayilearned,You learn something new every day; what did yo...,41M\n members,https://www.reddit.com/r/todayilearned/


In [4]:
# sanity check 
len(top500['Users'])

500

In [10]:
# remove unecessary characters in Users

users_list = top500['Users'].to_list()

cleaned_user_list = []

for entry in users_list: 
    match = re.search(r'^(\d+(?:\.\d+)?)', entry)
    if match: 
        cleaned_user_list.append(match.group(1))
    else:
        print(entry)

In [7]:
# sanity check
len(cleaned_user_list)

500

In [9]:
# convert cleaned list back to series

top500['Users'] = pd.Series(cleaned_user_list)
top500.head()

,Rank,Title,Description,Users,Title_URL
0,1,r/funny,Reddit's largest humor depository,67,https://www.reddit.com/r/funny/
1,2,r/AskReddit,r/AskReddit is the place to ask and answer tho...,58,https://www.reddit.com/r/AskReddit/
2,3,r/worldnews,"A place for major news from around the world, ...",47,https://www.reddit.com/r/worldnews/
3,4,r/gaming,The Number One Gaming forum on the Internet.,47,https://www.reddit.com/r/gaming/
4,5,r/todayilearned,You learn something new every day; what did yo...,41,https://www.reddit.com/r/todayilearned/


In [11]:
# normalize or clean title for subsquent verification

title_list = top500['Title'].to_list()

cleaned_title_list = []

for entry in title_list: 
    match = re.search(r'^r/(\S+)', entry)
    if match: 
        cleaned_title_list.append(match.group(1))
    else:
        print(entry)

In [12]:
# sanity check 
len(cleaned_title_list)

500

In [13]:
# convert cleaned title list to series

top500['Title'] = pd.Series(cleaned_title_list)
top500.head()

,Rank,Title,Description,Users,Title_URL
0,1,funny,Reddit's largest humor depository,67,https://www.reddit.com/r/funny/
1,2,AskReddit,r/AskReddit is the place to ask and answer tho...,58,https://www.reddit.com/r/AskReddit/
2,3,worldnews,"A place for major news from around the world, ...",47,https://www.reddit.com/r/worldnews/
3,4,gaming,The Number One Gaming forum on the Internet.,47,https://www.reddit.com/r/gaming/
4,5,todayilearned,You learn something new every day; what did yo...,41,https://www.reddit.com/r/todayilearned/


In [25]:
# subset of subreddit lowered 
subreddit_set = set(s.lower() for s in cleaned_title_list)

In [16]:
# check data 
# relative path - 'folder/file_name.zst'

# file_path = "submissions/RS_2024-03.zst"
file_path = "comments/RC_2024-03.zst"

with open(file_path, "rb") as f:
    dctx = zstd.ZstdDecompressor()

    with dctx.stream_reader(f) as reader:
        text_stream = io.TextIOWrapper(reader, encoding="utf-8")

        for i, line in enumerate(text_stream):
            data = json.loads(line)
            print(data)
            
            if i == 1:   # stop after 1 iteration
                break

{'_meta': {'retrieved_2nd_on': 1709380806}, 'all_awardings': [], 'approved_at_utc': None, 'approved_by': None, 'archived': False, 'associated_award': None, 'author': 'mouseat9', 'author_flair_background_color': None, 'author_flair_css_class': None, 'author_flair_richtext': [], 'author_flair_template_id': None, 'author_flair_text': None, 'author_flair_text_color': None, 'author_flair_type': 'text', 'author_fullname': 't2_9he7to31', 'author_is_blocked': False, 'author_patreon_flair': False, 'author_premium': False, 'awarders': [], 'banned_at_utc': None, 'banned_by': None, 'body': 'Bro it’s ok.  I teach students with mild  to severe  autism.   Children with autism can be a beautiful thing to behold.   But the very severe needs heavy socialization and lot of work.  I’ve seen men and women who are experts and dealing with severe autism has destroyed many families.  The single mothers who have to face this alone usually stay single, because of the enormous burden.   To tell you the truth, ev

In [28]:
# zst decompresor and streamer
class RedditZSTExtractor:
    """
    Extracts comments or submissions from Reddit Pushshift ZST files
    and streams them directly to CSV.
    """
    def __init__(self, outdir="data", bot_users=None):
        self.outdir = outdir
        os.makedirs(outdir, exist_ok=True)
        self.bot_users = bot_users or {"[deleted]", "AutoModerator"}

    def extract_to_csv(self, file_path, data_type="comments", subreddit_set=None):
        """
        Streams a ZST file to CSV.

        Parameters:
        - file_path: path to the ZST file
        - data_type: "comments" or "submissions" (used for CSV column names)
        - subreddit_set: set of subreddits to filter (optional)
        """
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"{file_path} does not exist")

        month = self._extract_month(file_path)
        output_csv = f"{self.outdir}/{data_type}_2024_{month}_full.csv"

        if os.path.exists(output_csv):
            print(f"File already exists: {output_csv}")
            return output_csv

        # determine column name
        count_col = "count_comments" if data_type == "comments" else "count_posts"

        with open(output_csv, "w", newline="", encoding="utf-8") as f_out:
            writer = csv.writer(f_out)
            writer.writerow(["author", "subreddit", count_col])  # header

            with open(file_path, "rb") as f_zst:
                dctx = zstd.ZstdDecompressor()
                with dctx.stream_reader(f_zst, read_size=2**24) as reader:
                    text_stream = io.TextIOWrapper(reader, encoding="utf-8", errors="replace")

                    for line in text_stream:
                        try:
                            data = orjson.loads(line)
                            sub = data.get("subreddit", "").lower()
                            user = data.get("author")

                            if subreddit_set and sub not in subreddit_set:
                                continue
                            if user in self.bot_users:
                                continue

                            writer.writerow([user, sub, 1])

                        except Exception:
                            continue

        print(f"All filtered {data_type} streamed to {output_csv}")
        return output_csv

    @staticmethod
    def _extract_month(file_path):
        """
        Tries to parse the month from the file name assuming format RC/RS_YYYY-MM.zst
        """
        import re
        m = re.search(r"_(\d{4})-(\d{2})\.zst$", file_path)
        if m:
            year, month_num = m.groups()
            # Convert numeric month to name (optional)
            import calendar
            month_name = calendar.month_name[int(month_num)].lower()
            return month_name
        return "unknown"

In [29]:
# initialize
extractor = RedditZSTExtractor(outdir="data")

# extract comments
comments_csv = extractor.extract_to_csv(
    file_path="comments/RC_2024-03.zst",
    data_type="comments",
    subreddit_set=subreddit_set
)

File already exists: data/comments_2024_march_full.csv


In [30]:
# extract submissions
submissions_csv = extractor.extract_to_csv(
    file_path="submissions/RS_2024-03.zst",
    data_type="submissions",
    subreddit_set=subreddit_set
)

File already exists: data/submissions_2024_march_full.csv
